# release16_constrained_checkpoint_repair_screen_v1

Executable model-selection screen for the latest release16 short-training failure.

## Governance

- Canonical baseline: `soxlbacktest_qlib_walkforward_baseline_v1`
- Direct comparison baseline: `release16_drawdown_drift_short_training_screen_v1`
- Role: `diagnose` and target-revision handoff, not promotion
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`
- Acceptance criteria: every fold must satisfy `relative_return >= +30pp` and `MaxDD < 20%`
- Changed layer: `model_selection`, `reward`, `checkpoint_selection`
- External reference: repo-owned constrained checkpoint selector and Qlib official replay evidence

## Historical Context

Closest success case: `release16_nonconvex_checkpoint_objective_score_handoff_v1` reached a much smaller residual gap but still required official replay training objective handoff.

Closest failure case: `release16_drawdown_drift_short_training_screen_v1` improved drift gates but failed all selected fold gates and cannot retain an actor.

This run is not a duplicate because it applies the hard MaxDD <20% / +30pp selector to every short-training checkpoint candidate and converts the outcome into a concrete next training objective contract.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = next(
    path for path in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
    if (path / "src" / "soxl_quant").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.constrained_checkpoint_selector import (
    ConstrainedCheckpointSelectorConfig,
    render_constrained_checkpoint_selection_review,
    score_constrained_checkpoints,
    summarize_constrained_checkpoint_selection,
)

EXPERIMENT = "release16_constrained_checkpoint_repair_screen_v1"
SOURCE_EXPERIMENT = "release16_drawdown_drift_short_training_screen_v1"
OUT = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE = ROOT / "runtime" / "model_selection" / "card" / SOURCE_EXPERIMENT
OUT.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
REQUIRED_WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]
CREATED_AT = datetime.now(timezone.utc).isoformat()

assert (SOURCE / "checkpoint_candidate_metrics.csv").exists(), SOURCE


## Load Short-Training Candidate Metrics

In [2]:
candidate_metrics = pd.read_csv(SOURCE / "checkpoint_candidate_metrics.csv")
selection_by_fold = pd.read_csv(SOURCE / "checkpoint_selection_by_fold.csv")
source_summary = json.loads((SOURCE / "summary.json").read_text(encoding="utf-8"))
source_selection = json.loads((SOURCE / "selection.json").read_text(encoding="utf-8"))

renamed = candidate_metrics.rename(
    columns={
        "fold": "window",
        "test_relative_return": "relative_return",
        "test_max_dd": "max_dd",
    }
)
for optional_column in ("test_avg_pos", "test_trade_count"):
    if optional_column not in renamed.columns:
        renamed[optional_column] = 0
gate_metrics = renamed[["candidate", "window", "relative_return", "max_dd", "test_avg_pos", "test_trade_count"]].copy()

print(f"candidate_rows={len(candidate_metrics)} windows={sorted(gate_metrics['window'].unique())}")
display(gate_metrics.head())


candidate_rows=12 windows=['wf_2020', 'wf_2022', 'wf_2023_2024']


,candidate,window,relative_return,max_dd,test_avg_pos,test_trade_count
0,ppo_retained_iter_4,wf_2020,-262.992258,20.937537,0.388424,0
1,ppo_selected_best,wf_2020,-262.992258,20.937537,0.388424,0
2,bc_initialization,wf_2020,-264.728894,20.918629,0.391214,0
3,ppo_retained_iter_0_bc_equivalent,wf_2020,-264.728894,20.918629,0.391214,0
4,bc_initialization,wf_2022,55.748206,26.832912,0.162569,0


## Apply Hard Constrained Selector

The selector ranks candidates by hard fold failures first. A candidate cannot be promoted unless every required fold satisfies both return and drawdown gates.

In [3]:
config = ConstrainedCheckpointSelectorConfig(
    target_relative_return_pp=TARGET_RELATIVE_RETURN_PP,
    target_max_dd_pct=TARGET_MAX_DD_PCT,
    max_dd_gap_weight=10.0,
)
scored = score_constrained_checkpoints(gate_metrics, config=config)
selector_summary = summarize_constrained_checkpoint_selection(scored)

score_path = OUT / "constrained_checkpoint_scores.csv"
gate_path = OUT / "gate_metrics.csv"
scored.to_csv(score_path, index=False)
gate_metrics.to_csv(gate_path, index=False)

print(selector_summary)
display(scored[[
    "candidate",
    "selector_gate_pass",
    "constrained_selector_loss",
    "window_count",
    "gate_pass_count",
    "failed_window_count",
    "relative_failure_count",
    "maxdd_failure_count",
    "min_relative_return",
    "max_max_dd",
    "relative_gap_sum",
    "max_dd_gap_sum",
]])


{'decision': 'constrained_checkpoint_selector_no_promotion', 'selected_candidate': 'bc_initialization', 'selected_constrained_selector_loss': 701.4843361939411, 'selected_constraint_gap_loss': 401.48433619394115, 'selected_failed_window_count': 3, 'selected_relative_failure_count': 1, 'selected_maxdd_failure_count': 3, 'selected_min_relative_return': -264.72889421266575, 'selected_max_max_dd': 26.8329120183762, 'selector_gate_pass': False, 'promotion_allowed': False, 'replay_review_required': False, 'next_mechanism': 'generate_reward_constrained_short_training_candidate', 'acceptance_criteria': {'target_relative_return_pp': 30.0, 'target_max_dd_pct': 20.0, 'all_windows_must_pass': True}}


,candidate,selector_gate_pass,constrained_selector_loss,window_count,gate_pass_count,failed_window_count,relative_failure_count,maxdd_failure_count,min_relative_return,max_max_dd,relative_gap_sum,max_dd_gap_sum
0,bc_initialization,False,701.484336,3,0,3,1,3,-264.728894,26.832912,294.728894,10.675544
1,ppo_retained_iter_0_bc_equivalent,False,701.484336,3,0,3,1,3,-264.728894,26.832912,294.728894,10.675544
2,ppo_retained_iter_4,False,877.418485,3,0,3,2,3,-262.992258,37.397618,295.629835,28.178865
3,ppo_selected_best,False,877.418485,3,0,3,2,3,-262.992258,37.397618,295.629835,28.178865


## Fold-Level Readout for Selected Diagnostic Candidate

In [4]:
top_candidate = str(selector_summary["selected_candidate"])
top_rows = candidate_metrics[candidate_metrics["candidate"].astype(str).eq(top_candidate)].copy()

selected_by_fold: dict[str, dict[str, object]] = {}
for row in top_rows.sort_values("fold").to_dict("records"):
    fold = str(row["fold"])
    selected_by_fold[fold] = {
        "selected_candidate": top_candidate,
        "selected_kind": str(row.get("candidate_kind") or ""),
        "selected_iter": int(row.get("iter", 0) or 0),
        "selected_relative_return": float(row["test_relative_return"]),
        "selected_max_dd": float(row["test_max_dd"]),
        "selected_avg_pos": float(row.get("test_avg_pos", 0.0) or 0.0),
        "selected_trade_count": int(row.get("test_trade_count", row.get("trade_count", 0)) or 0),
        "selected_candidate_gate": bool(row.get("candidate_gate", False)),
        "selected_target_gate": bool(row.get("target_gate", False)),
        "selected_drift_gate": bool(row.get("drift_gate", False)),
        "selected_daily_path": str(row.get("daily_path", "")),
        "relative_gap_to_target_pp": max(0.0, TARGET_RELATIVE_RETURN_PP - float(row["test_relative_return"])),
        "max_dd_gap_to_target_pp": max(0.0, float(row["test_max_dd"]) - TARGET_MAX_DD_PCT),
    }

fold_readout = pd.DataFrame.from_records(
    [{"fold": fold, **payload} for fold, payload in selected_by_fold.items()]
).sort_values("fold")
fold_readout.to_csv(OUT / "selected_candidate_fold_readout.csv", index=False)
display(fold_readout)


,fold,selected_candidate,selected_kind,selected_iter,selected_relative_return,selected_max_dd,selected_avg_pos,selected_trade_count,selected_candidate_gate,selected_target_gate,selected_drift_gate,selected_daily_path,relative_gap_to_target_pp,max_dd_gap_to_target_pp
0,wf_2020,bc_initialization,bc_initialization,0,-264.728894,20.918629,0.391214,0,False,False,True,runtime/model_selection/card/release16_drawdow...,294.728894,0.918629
1,wf_2022,bc_initialization,bc_initialization,0,55.748206,26.832912,0.162569,0,False,False,True,runtime/model_selection/card/release16_drawdow...,0.000000,6.832912
2,wf_2023_2024,bc_initialization,bc_initialization,0,32.133653,22.924003,0.185929,0,False,False,True,runtime/model_selection/card/release16_drawdow...,0.000000,2.924003


## Objective Contract

No checkpoint candidate can be promoted. The next useful step is a new reward-constrained short-training candidate whose objective is optimized directly against the fold-level gaps shown above.

In [5]:
objective_contract = {
    "experiment": EXPERIMENT,
    "created_at": CREATED_AT,
    "decision": "reward_constrained_short_training_candidate_ready",
    "source_experiment": SOURCE_EXPERIMENT,
    "selected_diagnostic_candidate": top_candidate,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "required_wf_windows": REQUIRED_WINDOWS,
    "objective_terms": [
        "walkforward_relative_floor_gap_loss",
        "walkforward_drawdown_gap_loss",
        "fold_local_checkpoint_constraint_loss",
        "official_three_fold_replay_gate",
    ],
    "candidate_generation_rules": {
        "train_from": "fold_local_bc_or_sparse_release_bc_warm_start",
        "select_with": "constrained_checkpoint_selector_maxdd20_relative30",
        "reject_if_any_fold_max_dd_gte_pct": TARGET_MAX_DD_PCT,
        "reject_if_any_fold_relative_return_lt_pp": TARGET_RELATIVE_RETURN_PP,
        "retain_actor_only_after_official_three_fold_replay": True,
    },
    "next_action": "train_reward_constrained_short_training_candidate_with_relative_floor_and_drawdown_gap_loss_then_official_three_fold_replay",
}

selection = {
    "experiment": EXPERIMENT,
    "created_at": CREATED_AT,
    "decision": selector_summary["decision"],
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": SOURCE_EXPERIMENT,
    "role": "diagnose_constrained_checkpoint_repair",
    "changed_layer": ["model_selection", "reward", "checkpoint_selection"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "required_wf_windows": REQUIRED_WINDOWS,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "selector_gate_pass": bool(selector_summary["selector_gate_pass"]),
    "selected_candidate": top_candidate,
    "selected_by_fold": selected_by_fold,
    "failed_folds": [fold for fold, payload in selected_by_fold.items() if payload["relative_gap_to_target_pp"] > 0 or payload["max_dd_gap_to_target_pp"] > 0],
    "next_training_allowed_after_target_revision": True,
    "next_mechanism": objective_contract["next_action"],
    "objective_contract": "objective_contract.json",
    "source_experiment": SOURCE_EXPERIMENT,
    "source_path": str((SOURCE / "summary.json").relative_to(ROOT)),
    "sidecars": [
        "gate_metrics.csv",
        "constrained_checkpoint_scores.csv",
        "selected_candidate_fold_readout.csv",
        "objective_contract.json",
        "constrained_checkpoint_review.md",
        "selection.json",
        "summary.json",
    ],
}

summary = {
    **selection,
    "selected_constrained_selector_loss": selector_summary["selected_constrained_selector_loss"],
    "selected_constraint_gap_loss": selector_summary["selected_constraint_gap_loss"],
    "selected_failed_window_count": selector_summary["selected_failed_window_count"],
    "selected_relative_failure_count": selector_summary["selected_relative_failure_count"],
    "selected_maxdd_failure_count": selector_summary["selected_maxdd_failure_count"],
    "selected_min_relative_return": selector_summary["selected_min_relative_return"],
    "selected_max_max_dd": selector_summary["selected_max_max_dd"],
    "worst_wf_max_drawdown_pct": float(max(payload["selected_max_dd"] for payload in selected_by_fold.values())),
    "min_wf_relative_return_pp": float(min(payload["selected_relative_return"] for payload in selected_by_fold.values())),
    "max_drawdown_gap_to_target_pp": float(max(payload["max_dd_gap_to_target_pp"] for payload in selected_by_fold.values())),
    "max_relative_gap_to_target_pp": float(max(payload["relative_gap_to_target_pp"] for payload in selected_by_fold.values())),
}

(OUT / "objective_contract.json").write_text(json.dumps(objective_contract, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(OUT / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
review_summary = {**selector_summary, "next_mechanism": objective_contract["next_action"]}
review = render_constrained_checkpoint_selection_review(scored, review_summary)
(OUT / "constrained_checkpoint_review.md").write_text(review, encoding="utf-8")

print(json.dumps({key: summary[key] for key in ["experiment", "decision", "selected_candidate", "selected_constrained_selector_loss", "selected_failed_window_count", "promotion_allowed", "next_training_allowed_after_target_revision", "next_mechanism"]}, indent=2, sort_keys=True))


{
  "decision": "constrained_checkpoint_selector_no_promotion",
  "experiment": "release16_constrained_checkpoint_repair_screen_v1",
  "next_mechanism": "train_reward_constrained_short_training_candidate_with_relative_floor_and_drawdown_gap_loss_then_official_three_fold_replay",
  "next_training_allowed_after_target_revision": true,
  "promotion_allowed": false,
  "selected_candidate": "bc_initialization",
  "selected_constrained_selector_loss": 701.4843361939411,
  "selected_failed_window_count": 3
}


## Conclusion

The constrained selector found no promotable release16 short-training checkpoint under `relative_return >= +30pp` and `MaxDD < 20%` across all required walk-forward folds. The closest diagnostic candidate is still `bc_initialization`, but it fails every fold, with worst MaxDD `26.83%` and the same large `wf_2020` relative-return gap seen in the source card.

Decision: no actor retention and no promotion. The next aligned action is to train a new reward-constrained short-training candidate using explicit walk-forward relative-floor and drawdown-gap losses, then run official three-fold replay before any actor can be retained.